In [1]:
import gymnasium as gym

import numpy as np
import time
import matplotlib
import matplotlib.pyplot as plt
import random 
import numpy as np
from collections import defaultdict
#import tensorflow as tf
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Input
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
from collections import deque
 
 

env=gym.make('CartPole-v1')#,render_mode='human')

(state,_)=env.reset()

In [2]:
class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),  # Increased size for hidden layers
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, states):
        input_list = np.array(states,dtype=np.float32)
        x=torch.from_numpy(input_list).float()
        return self.fc(x)
    

In [3]:
class DQAgent:

    def __init__(self, state_dim, action_dim, gamma=0.99, lr=0.001, batch_size=64, buffer_size=10000):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.gamma = gamma
        self.lr = lr
        self.batch_size = batch_size
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.memory = deque(maxlen=buffer_size)

        # Models
        self.model = DQN(state_dim, action_dim)
        self.target_model = DQN(state_dim, action_dim)
        self.update_target_model()

        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.loss_fn = nn.SmoothL1Loss() 
        
    def get_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, self.action_dim - 1)
        else:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)
            with torch.no_grad():
                q_values = self.model(state_tensor)
            return torch.argmax(q_values).item()
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())
            
    def train(self): 
        
        if len(self.memory) < self.batch_size:
            return  # Not enough samples to train

        batch = random.sample(self.memory, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.FloatTensor(states)
        actions = torch.LongTensor(actions)
        rewards = torch.FloatTensor(rewards)
        next_states = torch.FloatTensor(next_states)
        dones = torch.FloatTensor(dones)

        # Compute target Q-values
        with torch.no_grad():
            next_q_values = self.target_model(next_states)
            max_next_q_values = next_q_values.max(dim=1)[0]
            target_q_values = rewards + (1 - dones) * self.gamma * max_next_q_values

        # Compute current Q-values
        q_values = self.model(states)
        current_q_values = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)

        # Compute loss
        loss = self.loss_fn(current_q_values, target_q_values)

        # Backpropagation
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        
        

In [4]:
def treatState(state,level,goal):
    tmp_level= min(level/500,1)
    state =np.append(state,tmp_level)
    return state

In [5]:

def KL(a,b):
        
        return np.sum(a*np.log(a/b))


In [6]:
def changeReward(reward,done,num,goal):
    result=num
    if num>goal:
        return -(reward+0.01*(num-goal))
    return 1 + 0.01 * num
    
    """if result < goal-10:
        reward=1
    elif result > goal+10:
        reward=-1
    else:
        reward=goal - abs(goal-result)
    return reward"""
    if done:
        if num>goal:
             return reward-0.01*(num-goal)#+10
        return reward-0.01*(goal-num)#-10
    else:
        if num>goal:
             return -(reward+0.01*(num-goal))
        return 1 + 0.01 * num

In [7]:
goaldistr=np.random.normal(loc=50,scale=10,size=1000)


## test

In [8]:
episodeNumber=30000
timeSteps=1000
episodesResults=[]
episodesTestResults=[]
episodesTestResultsDeviation=[]
KLdivergenceResult=[]
#goaldistr=np.random.normal(loc=100,scale=3,size=100)
numberOfTest=50
agent = DQAgent(5, 2,batch_size=512)
goal=50


In [9]:
for episodeIndex in range(1,episodeNumber+1):
    state,_=env.reset()
    state=treatState(state,0,goal)
    
    
   
    for timeIndex in range(timeSteps+1):
        action=agent.get_action(state)
        next_state, reward, done, _, _ = env.step(action)
        next_state=treatState(next_state,timeIndex+1,goal)
        agent.store_transition(state, action, changeReward(reward,done,timeIndex,goal), next_state, done)
        #agent.store_transition(state, action, KL(np.append(episodesResults,timeIndex),goaldistr), next_state, done)
        state=next_state
        if timeIndex %10 ==0:
            agent.train()
        if (done or timeIndex==timeSteps-1 ):
            
            episodesResults.append(timeIndex)
            break
    
    if(episodeIndex %100 ==0):
        agent.update_target_model()
    if(episodeIndex %250 ==0):
        print("episode : ",episodeIndex)
    if (episodeIndex %1000 ==0):
        KLdivergenceResult.append(KL(goaldistr, episodesResults[-1000:]))
    if(episodeIndex %500 ==0):
        tmp_epsilon=agent.epsilon
        agent.epsilon=0
        testResultList=[]
        for episodeIndex in range(numberOfTest):
            state,_=env.reset()
            state=treatState(state,0,goal)
            for timeIndex in range(timeSteps):
                action=agent.get_action(tuple(state))
                next_state, reward, done, _, _ = env.step(action)
                next_state=treatState(next_state,timeIndex+1,goal)
                state=next_state
                if (done):
                    testResultList.append(timeIndex)
                    break
        episodesTestResults.append(sum(testResultList)/numberOfTest)
        episodesTestResultsDeviation.append(np.std(episodesTestResults))
        agent.epsilon=tmp_epsilon

env.close()

C:\Users\mathi\AppData\Local\Temp\ipykernel_23188\2635204200.py:43: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  states = torch.FloatTensor(states)


episode :  250
episode :  500
episode :  750
episode :  1000
[-17722.978779211146]
episode :  1250
episode :  1500
episode :  1750
episode :  2000
[-17722.978779211146, -15900.396348535913]
episode :  2250
episode :  2500
episode :  2750
episode :  3000
[-17722.978779211146, -15900.396348535913, 53889.66158272791]
episode :  3250
episode :  3500
episode :  3750
episode :  4000
[-17722.978779211146, -15900.396348535913, 53889.66158272791, 11986.070535439607]
episode :  4250
episode :  4500
episode :  4750
episode :  5000
[-17722.978779211146, -15900.396348535913, 53889.66158272791, 11986.070535439607, -111.56235992571669]
episode :  5250
episode :  5500
episode :  5750
episode :  6000
[-17722.978779211146, -15900.396348535913, 53889.66158272791, 11986.070535439607, -111.56235992571669, -4606.852029180066]
episode :  6250
episode :  6500
episode :  6750
episode :  7000
[-17722.978779211146, -15900.396348535913, 53889.66158272791, 11986.070535439607, -111.56235992571669, -4606.85202918006

In [10]:
goal12LearnerEp=episodesResults
%store goal12LearnerEp

DQN6episodesTestResults=episodesTestResults
%store DQN6episodesTestResults

DQN6episodesTestResultsDeviation=episodesTestResultsDeviation
%store DQN6episodesTestResultsDeviation

DQN6KL=KLdivergenceResult
%store DQN6KL

Stored 'goal12LearnerEp' (list)
Stored 'DQN6episodesTestResults' (list)
Stored 'DQN6episodesTestResultsDeviation' (list)
Stored 'DQN6KL' (list)


In [11]:
print(episodesTestResultsDeviation)
print(episodesTestResults)
print(len(episodesResults))

[0.0, 42.52, 60.87659210785922, 58.44185486447192, 64.84183637128116, 65.69320783426211, 63.851251629814406, 59.84002590240081, 56.55236087899121, 53.69244002650653, 51.194969987940375, 49.090149356962534, 47.189243333523926, 45.47723377511801, 43.945825113301595, 42.551709113001564, 41.409303803885564, 40.68824262184742, 39.61240028016074, 38.62586712295272, 37.695020331910285, 36.8531225899258, 36.087419399860096, 35.32813042471333, 34.61435777015082, 33.95943142692817, 33.413179939807975, 32.90374139758553, 40.240325146491585, 43.15765817969687, 46.070857094969924, 45.345906784508955, 44.72344561929953, 44.089561118863266, 43.520396096636844, 43.05677719772255, 42.50694509832956, 42.01198660572429, 41.49299095298931, 41.02455883675046, 41.11409143993806, 40.62197459727193, 40.16169502821327, 39.707998098828014, 39.30658260707631, 39.177800820447466, 39.5228695878949, 39.7987872163071, 39.400521513598314, 39.078350311137754, 39.09258032573974, 38.82403978893161, 38.456031184929394, 3